# Part 2 : Dashboard for surfers !


## Step 1 : Data extraction

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
from datetime import datetime


Extract the data

In [2]:
# Étape 1 : Récupérer le contenu de la page
url = "https://www.surf-report.com/meteo-surf/lacanau-s1043.html"

headers = {
    'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:146.0) Gecko/20100101 Firefox/146.0"
}

print("🌊 Récupération des données de Surf-Report...")
response = requests.get(url, headers=headers)
response.encoding = 'utf-8'

print(f"✅ Status: {response.status_code}")

🌊 Récupération des données de Surf-Report...


✅ Status: 200


Put the extracted data in a dataframe

In [3]:

# Étape 2 : Parser le HTML
soup = BeautifulSoup(response.content, 'html.parser')

# Étape 3 : Trouver le tableau de prévisions
# Chercher toutes les divs ou sections contenant les prévisions par jour
days_sections = soup.find_all('div', class_=re.compile(r'jour|day|forecast'))

data = []

print("\n📊 Extraction des prévisions depuis le HTML...")

# Si la structure ci-dessus ne fonctionne pas, extraire directement depuis le code source
script_content = response.text

# Pattern pour extraire les données JSON
pattern = r'\["(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})"\]=>[\s\S]*?object\(stdClass\)#\d+ \(\d+\) \{([^}]+(?:\{[^}]*\}[^}]*)*)\}'
matches = re.finditer(pattern, script_content)

# Fonction pour extraire une valeur
def extract_field(field_name, text):
    pattern = rf'\["{field_name}"\]=>\s*string\(\d+\)\s*"([^"]*)"'
    m = re.search(pattern, text)
    return m.group(1) if m else None

# Fonction pour convertir les degrés en code de direction
def degres_vers_code_direction(degres_str):
    if not degres_str:
        return "N/A"
    try:
        degres = float(degres_str)
        directions = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                     'S', 'SSO', 'SO', 'OSO', 'O', 'ONO', 'NO', 'NNO']
        index = round(degres / 22.5) % 16
        return directions[index]
    except:
        return "N/A"

# Étape 4 : Pour chaque horaire, extraire les données
for match in matches:
    try:
        date_time_str = match.group(1)
        forecast_block = match.group(2)
        
        # Parser la date et l'heure
        date_obj = datetime.strptime(date_time_str, '%Y-%m-%d %H:%M:%S')
        
        # Formater la date en français
        jours_fr = {
            'Monday': 'Lundi', 'Tuesday': 'Mardi', 'Wednesday': 'Mercredi',
            'Thursday': 'Jeudi', 'Friday': 'Vendredi', 'Saturday': 'Samedi', 'Sunday': 'Dimanche'
        }
        mois_fr = {
            'January': 'Janvier', 'February': 'Février', 'March': 'Mars', 'April': 'Avril',
            'May': 'Mai', 'June': 'Juin', 'July': 'Juillet', 'August': 'Août',
            'September': 'Septembre', 'October': 'Octobre', 'November': 'Novembre', 'December': 'Décembre'
        }
        
        jour_en = date_obj.strftime('%A')
        mois_en = date_obj.strftime('%B')
        jour_num = date_obj.strftime('%d')
        
        # Format : "Samedi 22 Octobre"
        day = f"{jours_fr[jour_en]} {jour_num} {mois_fr[mois_en]}"
        hour = date_obj.strftime('%H:%M')
        
        # Extraire les valeurs
        houle = extract_field('houle', forecast_block)
        houle_max = extract_field('houleMax', forecast_block)
        vent_moyen = extract_field('ventMoyen', forecast_block)
        direction_vent_deg = extract_field('directionVent', forecast_block)
        
        # Formater la taille des vagues : "0.8 - 1.3"
        if houle and houle_max:
            waves_size = f"{houle} - {houle_max}"
        else:
            waves_size = "N/A"
        
        # Formater la vitesse du vent : "\n3\n\n"
        if vent_moyen:
            wind_speed = f"\\n{vent_moyen}\\n\\n"
        else:
            wind_speed = "N/A"
        
        # Convertir les degrés en code de direction (N, S, E, O, etc.)
        direction_code = degres_vers_code_direction(direction_vent_deg)
        
        # Ajouter aux données
        data.append({
            'day': day,
            'hour': hour,
            'waves_size': waves_size,
            'wind_speed': wind_speed,
            'direction_vent_deg': direction_code
        })
        
    except Exception as e:
        print(f"⚠️ Erreur lors du traitement d'une entrée: {e}")
        continue

# Étape 5 : Créer le DataFrame
if len(data) == 0:
    print("❌ Aucune donnée extraite. Vérifiez la structure de la page.")
else:
    df = pd.DataFrame(data)
    
    # Limiter à 7 jours
    dates_uniques = df['day'].unique()
    nb_jours = min(7, len(dates_uniques))
    df_7j = df[df['day'].isin(dates_uniques[:nb_jours])]
    
    # Étape 6 : Appliquer le mapping des directions longues
    directions_longues = {
        'N': 'Orientation vent Nord',
        'NNE': 'Orientation vent Nord Est',
        'NE': 'Orientation vent Nord Est',
        'ENE': 'Orientation vent Est Nord Est',
        'E': 'Orientation vent Est',
        'ESE': 'Orientation vent Est Sud Est',
        'SE': 'Orientation vent Sud Est',
        'SSE': 'Orientation vent Sud Est',
        'S': 'Orientation vent Sud',
        'SSO': 'Orientation vent Sud Ouest',
        'SO': 'Orientation vent Sud Ouest',
        'OSO': 'Orientation vent Ouest Sud Ouest',
        'O': 'Orientation vent Ouest',
        'ONO': 'Orientation vent Ouest Nord Ouest',
        'NO': 'Orientation vent Nord Ouest',
        'NNO': 'Orientation vent Nord Ouest',
        'N/A': 'N/A'
    }
    
    # Créer la colonne wind_direction avec le mapping
    df_7j['wind_direction'] = df_7j['direction_vent_deg'].map(directions_longues)
    
    # Réorganiser les colonnes dans l'ordre souhaité
    df_7j = df_7j[['day', 'hour', 'waves_size', 'wind_speed', 'wind_direction']]
    
    # Réinitialiser l'index pour commencer à 0
    df_7j = df_7j.reset_index(drop=True)
    
    


📊 Extraction des prévisions depuis le HTML...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_44324\1540084619.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_7j['wind_direction'] = df_7j['direction_vent_deg'].map(directions_longues)


In [4]:
df_7j["wind_direction"].unique()

array(['Orientation vent Ouest Nord Ouest', 'Orientation vent Ouest',
       'Orientation vent Ouest Sud Ouest', 'Orientation vent Sud Ouest',
       'Orientation vent Nord Ouest', 'Orientation vent Nord',
       'Orientation vent Est Nord Est', 'Orientation vent Sud Est',
       'Orientation vent Sud', 'N/A'], dtype=object)

In [5]:
df_7j.head()

,day,hour,waves_size,wind_speed,wind_direction
0,Jeudi 08 Janvier,01:00,2.6 - 4.0,\n39\n\n,Orientation vent Ouest Nord Ouest
1,Jeudi 08 Janvier,02:00,2.7 - 4.1,\n40\n\n,Orientation vent Ouest Nord Ouest
2,Jeudi 08 Janvier,03:00,2.8 - 4.2,\n41\n\n,Orientation vent Ouest Nord Ouest
3,Jeudi 08 Janvier,04:00,2.9 - 4.4,\n41\n\n,Orientation vent Ouest Nord Ouest
4,Jeudi 08 Janvier,05:00,3.0 - 4.5,\n42\n\n,Orientation vent Ouest


In [6]:
df_7j.shape

(174, 5)

Save the dataframe a csv file

In [7]:
# Sauvegarder le DataFrame avec index (0, 1, 2, 3...)
df_7j.to_csv('meteo_surf_lacanau.csv', index=True, encoding='utf-8-sig')

print(f" Fichier sauvegardé : meteo_surf_lacanau.csv ({len(df_7j)} lignes)")

 Fichier sauvegardé : meteo_surf_lacanau.csv (174 lignes)


Create a python library called “surf_scrap”(the user can import with the standard
python import command. In this library there will be only one function. This function
will allow the user to select a specific url (ex :
https://www.surf-report.com/meteo-surf/carcans-plage-s1013.html;
https://www.surf-report.com/meteo-surf/moliets-plage-centrale-s102799.html).
and save the related dataframe in csv file in the location, he/she wants)

Create a python (.py) script allowing to execute the library.


## Step 2 

In this last step, we create the following files:
- run_surf_scrap.py
- surf_dashboard.Rmd
- surf_dashboard.html
